# Etap 2. Transformacja danych

In [89]:
#Wgrywam dane z poprzedniego etapu
import pandas as pd
from pandas import unique

pd.set_option('display.max_columns', None)

customers_clean = pd.read_csv(r'C:\Users\l\Documents\DATA_SCIENCE_KURS\PROJEKT_PREDYCJI_SUBSKRYBCJI_NEWSLETTERA\PROJEKT\b_oprzadek\data\customers_clean.csv')
transactions_clean = pd.read_csv(r'C:\Users\l\Documents\DATA_SCIENCE_KURS\PROJEKT_PREDYCJI_SUBSKRYBCJI_NEWSLETTERA\PROJEKT\b_oprzadek\data\transactions_clean.csv')

### Konwersja danych

In [90]:
# CUSTOMERS
#Zmieniamy wiek z typu zmiennoprzecinkowego na liczbę całkowitą w celu oszczędzenia ramu
customers_clean["age"] = customers_clean["age"].astype(int)
#Zmieniamy płeć z tekstu na kategorię bo często się powtarza i zwolnimy pamięć
customers_clean["gender"] = customers_clean["gender"].astype("category")
#Zmieniamy state na kategorię w celu późniejszej oceny miejsca subsrybcji
customers_clean["state"] = customers_clean["state"].astype("category")
#Date z tekstu konwertujemy na daytime
customers_clean["signup_date"] = pd.to_datetime(customers_clean["signup_date"])
#Nr telefonu zmieniamy z liczby na tekst, aby uniknąć sytuacji z usuwaniem zer na początku lub końcu
customers_clean["phone_number"] = customers_clean["phone_number"].astype(str)
#Mapujemy subscribe czyli nasz target na 1 - Yes, 0 - No
customers_clean["subscribe"] = customers_clean["subscribe"].map({
    "Yes":1,
    "No":0
})

#TRANSACTIONS
#Date z tekstu konwertujemy na daytime
transactions_clean['transaction_date'] = pd.to_datetime(transactions_clean["transaction_date"])
#Zmieniamy payment_method oraz transaction_status z tekstu na kategorię
transactions_clean['payment_method'] = transactions_clean['payment_method'].astype("category")
transactions_clean['transaction_status'] = transactions_clean['transaction_status'].astype("category")

customers_clean.info()
transactions_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   customer_id   1000 non-null   str           
 1   name          1000 non-null   str           
 2   age           1000 non-null   int64         
 3   gender        1000 non-null   category      
 4   state         1000 non-null   category      
 5   signup_date   1000 non-null   datetime64[us]
 6   email         1000 non-null   str           
 7   phone_number  1000 non-null   str           
 8   subscribe     1000 non-null   int64         
dtypes: category(2), datetime64[us](1), int64(2), str(4)
memory usage: 57.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 8199 entries, 0 to 8198
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   transaction_id      8199 non-null   str           
 1   customer

## Budowa modelu

### Wybór istniejących wartości:

Z istniejących wartości na subskrybcję wpływ może mieć:
* wiek - 'age'
* płeć - 'gender'
* miejsce zamieszkania - 'state' (później analizowane)

### Feature engineering

Z tabeli transactions należy stworzyć cechy, które mogą mieć wpływ na subskrybcję:
* staz_klienta - klient zarejestrowany wczoraj zachowuje się inaczej niż klikuletni
* liczba_transakcji - ilość zakupów wpływa na to czy klient jest jednorazowy czy stały
* laczna_kwota - calkowita ilosc pieniedzy wydana w sklepie
* suma_rabatow - suma znizek moze miec wplyw na obecnosc klienta
* ilosc_unikalnych_produktow - ile różnych produktów klient kupił
* dni_od_ostatniej_transakcji - ile czasu mineło od ostatniego zakupu

In [91]:
#Wartosc pojedynczej transakcji:
transactions_clean["transaction_value"] = (transactions_clean["quantity"]*transactions_clean["unit_price"])
#Ilość dni od momentu rejestracji klienta (staż)
customers_clean["customer_tenture_days"] = (customers_clean['signup_date'].max() - customers_clean['signup_date']).dt.days
#customers_clean["customer_tenture_days"] = customers_clean["customer_tenture_days"]
#.astype(int)
#Agregacja transakcji do poziomu klienta
customer_transactions = transactions_clean.groupby("customer_id").agg(
    transaction_count = ('transaction_id','count'), #liczba transakcji na klienta
    total_spent = ('transaction_value','sum'), #calkowita suma wydana przez klienta
    avg_transaction_value = ('transaction_value','mean'), #średnia wartość transakcji
    total_quantity = ('quantity','sum'), #calkowita suma kupionych produktow
    avg_discount=("discount_applied", "mean"), #srednia znizka na klienta
    unique_products = ('product_id','nunique'),  #ilosc unikalnych produktow kupionych przez klienta
    last_transaction_date=("transaction_date", "max") #ostatnia dokonana transakcja
    )
#Zaokrąglam wartości średnie do 2 miejsc po przecinku
customer_transactions["avg_transaction_value"] = (customer_transactions["avg_transaction_value"].round(2))
customer_transactions["avg_discount"] = (customer_transactions["avg_discount"].round(2))


### Analiza kolumny 'state'

In [92]:
print(customers_clean['state'].nunique())
subsribe_state_value = customers_clean.groupby("state")["subscribe"].value_counts(normalize=True)
total_subscribers = customers_clean["subscribe"].value_counts(normalize=True)
print(subsribe_state_value)
print(total_subscribers)

35
state             subscribe
Aceh              1            0.787879
                  0            0.212121
Bali              1            0.769231
                  0            0.230769
Banten            1            0.833333
                                 ...   
Sumatera Selatan  0            0.240000
Sumatera Utara    1            0.636364
                  0            0.363636
unknown           1            0.660377
                  0            0.339623
Name: proportion, Length: 70, dtype: float64
subscribe
1    0.711
0    0.289
Name: proportion, dtype: float64


Kolumna state zawierała 35 kategorii. Analiza rozkładu zmiennej docelowej względem stanu wykazała, że udział subskrybentów w poszczególnych stanach jest zbliżony do udziału w całej populacji (71,1%). W związku z tym uznano, że cecha wnosi niewielką dodatkową informację, a jej kodowanie metodą One-Hot zwiększyłoby liczbę zmiennych bez wyraźnych korzyści.

### Kodowanie zmiennych kategorycznych

Zmienna kategoryczna gender została zakodowana metodą One-Hot Encoding, ponieważ jest to zmienna nominalna i nie posiadają naturalnego porządku. Zastosowano parametr drop_first=True, aby uniknąć redundancji zmiennych i ograniczyć współliniowość.

In [93]:
customers_clean = pd.get_dummies(
    customers_clean,
    columns=["gender"],
    drop_first=True
)

### Przygotowanie tabeli

Target subscribe.

Tabela od której określamy predykcję:
* age
* gender_male
* customer_tenture_days
* transaction_count
* total_spent
* avg_transaction_value
* total_quantity
* avg_discount
* unique_products
* last_transaction_date

In [94]:
#Merge całej tabeli customers_clean z customer_transactions
customers_features = customers_clean.merge(
    customer_transactions,
    on="customer_id",
    how = 'left'
)

customers_features.info()

#Wybór tylko cech do modelu
features = [
    "age",
    "gender_male",
    "customer_tenure_days",
    "transaction_count",
    "total_spent",
    "avg_transaction_value",
    "total_quantity",
    "unique_products",
    "avg_discount"
]
target = ['subscribe']

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   customer_id            1000 non-null   str           
 1   name                   1000 non-null   str           
 2   age                    1000 non-null   int64         
 3   state                  1000 non-null   category      
 4   signup_date            1000 non-null   datetime64[us]
 5   email                  1000 non-null   str           
 6   phone_number           1000 non-null   str           
 7   subscribe              1000 non-null   int64         
 8   customer_tenture_days  1000 non-null   int64         
 9   gender_Male            1000 non-null   bool          
 10  transaction_count      1000 non-null   int64         
 11  total_spent            1000 non-null   int64         
 12  avg_transaction_value  1000 non-null   float64       
 13  total_quantity 

### Eksport do etapu 3.

In [95]:
customers_features.to_csv(r'C:\Users\l\Documents\DATA_SCIENCE_KURS\PROJEKT_PREDYCJI_SUBSKRYBCJI_NEWSLETTERA\PROJEKT\b_oprzadek\data\customers_features.csv', index=False)